In [1]:
# RegMap – Sentence‑BERT Fine‑Tuning for NIST‑to‑HIPAA Mapping

In [2]:
import pandas as pd
import numpy as np
import random
import json
import datetime
from pathlib import Path

from sklearn.model_selection import train_test_split

from sentence_transformers import (
    SentenceTransformer,
    InputExample,
    losses,
    evaluation
)
from torch.utils.data import DataLoader
import torch

# ---------- Paths ----------
DATA_RAW = Path('../data/raw')
DATA_PROCESSED = Path('../data/processed')
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

MODEL_DIR = Path('../models/regmap-embedder')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ---------- Reproducibility ----------
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

C:\Users\stamp\AppData\Local\Temp\ipykernel_31532\3819609685.py:10: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import (
C:\Users\stamp\AppData\Local\Temp\ipykernel_31532\3819609685.py:10: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers import (


In [3]:
# Reload raw crosswalk
crosswalk_raw = pd.read_csv(DATA_RAW / 'nist_800_53_rev5_hipaa_crosswalk.csv')

# Clean column names (remove newlines / extra spaces)
crosswalk_raw.columns = (
    crosswalk_raw.columns
    .str.replace('\n', ' ', regex=False)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

# Rename to our standard names
crosswalk = crosswalk_raw.rename(columns={
    'Focal Document Element': 'nist_control_id',
    'Focal Document Element Description': 'nist_text',
    'Reference Document Element': 'hipaa_citation',
    'Reference Document Element Description': 'hipaa_text'
})

# Keep only the columns we need
crosswalk = crosswalk[['nist_control_id', 'nist_text', 'hipaa_citation', 'hipaa_text']]

# Replace empty strings with NaN and drop any row missing text
crosswalk.replace('', np.nan, inplace=True)
crosswalk.dropna(subset=['nist_text', 'hipaa_text', 'hipaa_citation'], inplace=True)

# Strip whitespace from all text columns
for col in ['nist_text', 'hipaa_text', 'hipaa_citation']:
    crosswalk[col] = crosswalk[col].astype(str).str.strip()

# Drop exact duplicate mappings (same control + same citation)
crosswalk = crosswalk.drop_duplicates(subset=['nist_control_id', 'hipaa_citation']).copy()

# All remaining rows are positive (true) mappings
crosswalk['label'] = 1

print(f"Total positive pairs: {len(crosswalk)}")
crosswalk.head()

Total positive pairs: 40


,nist_control_id,nist_text,hipaa_citation,hipaa_text,label
0,AC-01,"a. Develop, document, and disseminate to [Assi...",164.308(a)(3),Workforce Security: Implement policies and pro...,1
1,AC-01,"a. Develop, document, and disseminate to [Assi...",164.308(a)(3)(ii)(A),Authorization and/or Supervision (A): Implemen...,1
2,AC-01,"a. Develop, document, and disseminate to [Assi...",164.308(a)(4),Information Access Management: Implement polic...,1
3,AC-01,"a. Develop, document, and disseminate to [Assi...",164.308(a)(4)(ii)(B),Access Authorization (A): Implement policies a...,1
4,AC-01,"a. Develop, document, and disseminate to [Assi...",164.308(a)(4)(ii)(C),Access Establishment and Modification (A): Imp...,1


In [4]:
# Stratification not needed (all labels are 1), just simple random splits
train_val, test = train_test_split(
    crosswalk,
    test_size=0.15,
    random_state=SEED
)
train, val = train_test_split(
    train_val,
    test_size=0.15,   # ~12.75% of total for val
    random_state=SEED
)

print(f"Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")

Train: 28 | Val: 6 | Test: 6


In [5]:
def df_to_examples(df):
    examples = []
    for _, row in df.iterrows():
        # MultipleNegativesRankingLoss uses implicit positives:
        # the pair is the positive, all other pairs in batch are negatives.
        examples.append(InputExample(texts=[row['nist_text'], row['hipaa_text']]))
    return examples

train_examples = df_to_examples(train)
val_examples   = df_to_examples(val)
test_examples  = df_to_examples(test)

print(f"Train examples: {len(train_examples)}")
print(f"Val examples:   {len(val_examples)}")
print(f"Test examples:  {len(test_examples)}")

Train examples: 28
Val examples:   6
Test examples:  6


In [6]:
model = SentenceTransformer('all-MiniLM-L6-v2')
train_loss = losses.MultipleNegativesRankingLoss(model)
print("Model loaded with MultipleNegativesRankingLoss.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded with MultipleNegativesRankingLoss.


In [7]:
# Build a corpus from all unique HIPAA texts in the validation set
val_hipaa_texts = val['hipaa_text'].unique()
corpus = {i: text for i, text in enumerate(val_hipaa_texts)}

# For each validation query (NIST text), specify the relevant HIPAA citation(s)
queries = {}
relevant_docs = {}
qid = 0
for _, row in val.iterrows():
    query_text = row['nist_text']
    true_text = row['hipaa_text']
    # Find the corpus id of the true HIPAA text
    true_id = None
    for idx, ctext in corpus.items():
        if ctext == true_text:
            true_id = idx
            break
    if true_id is not None:
        queries[qid] = query_text
        relevant_docs[qid] = {true_id}
        qid += 1

ir_evaluator = evaluation.InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name="val-hipaa",
    show_progress_bar=True
)
print(f"Validation queries: {len(queries)}")

Validation queries: 6


In [9]:
# import sys
# !{sys.executable} -m pip install --upgrade "accelerate>=1.1.0"


In [8]:
BATCH_SIZE = 16          # larger batch helps MultipleNegativesRankingLoss
EPOCHS = 10
WARMUP_STEPS = int(len(train_examples) / BATCH_SIZE * 0.1)

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=EPOCHS,
    warmup_steps=WARMUP_STEPS,
    evaluator=ir_evaluator,
    evaluation_steps=500,
    output_path=str(MODEL_DIR),
    save_best_model=True,
    show_progress_bar=True
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

C:\Users\stamp\AppData\Local\Programs\Python\Python314\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss,Val-hipaa Cosine Accuracy@1,Val-hipaa Cosine Accuracy@3,Val-hipaa Cosine Accuracy@5,Val-hipaa Cosine Accuracy@10,Val-hipaa Cosine Precision@1,Val-hipaa Cosine Precision@3,Val-hipaa Cosine Precision@5,Val-hipaa Cosine Precision@10,Val-hipaa Cosine Recall@1,Val-hipaa Cosine Recall@3,Val-hipaa Cosine Recall@5,Val-hipaa Cosine Recall@10,Val-hipaa Cosine Ndcg@10,Val-hipaa Cosine Mrr@10,Val-hipaa Cosine Map@100
2,No log,No log,0.000000,0.833333,1.000000,1.000000,0.000000,0.277778,0.200000,0.100000,0.000000,0.833333,1.000000,1.000000,0.546607,0.394444,0.394444
4,No log,No log,0.166667,0.666667,1.000000,1.000000,0.166667,0.222222,0.200000,0.100000,0.166667,0.666667,1.000000,1.000000,0.596565,0.463889,0.463889
6,No log,No log,0.333333,0.500000,1.000000,1.000000,0.333333,0.166667,0.200000,0.100000,0.333333,0.500000,1.000000,1.000000,0.624701,0.505556,0.505556
8,No log,No log,0.333333,0.500000,1.000000,1.000000,0.333333,0.166667,0.200000,0.100000,0.333333,0.500000,1.000000,1.000000,0.624701,0.505556,0.505556
10,No log,No log,0.333333,0.500000,1.000000,1.000000,0.333333,0.166667,0.200000,0.100000,0.333333,0.500000,1.000000,1.000000,0.624701,0.505556,0.505556
12,No log,No log,0.333333,0.500000,1.000000,1.000000,0.333333,0.166667,0.200000,0.100000,0.333333,0.500000,1.000000,1.000000,0.624701,0.505556,0.505556
14,No log,No log,0.333333,0.500000,1.000000,1.000000,0.333333,0.166667,0.200000,0.100000,0.333333,0.500000,1.000000,1.000000,0.624701,0.505556,0.505556
16,No log,No log,0.333333,0.500000,1.000000,1.000000,0.333333,0.166667,0.200000,0.100000,0.333333,0.500000,1.000000,1.000000,0.624701,0.505556,0.505556
18,No log,No log,0.333333,0.500000,1.000000,1.000000,0.333333,0.166667,0.200000,0.100000,0.333333,0.500000,1.000000,1.000000,0.624701,0.505556,0.505556
20,No log,No log,0.333333,0.500000,1.000000,1.000000,0.333333,0.166667,0.200000,0.100000,0.333333,0.500000,1.000000,1.000000,0.624701,0.505556,0.505556


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

...

Corpus Chunks: 100%|██████████| 1/1 [00:00<00:00,  5.38it/s]


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:00<00:00,  5.65it/s]


In [10]:
# Load the best model saved during training
best_model = SentenceTransformer(str(MODEL_DIR))

# Build test corpus (all unique HIPAA texts in test set)
test_hipaa_texts = test['hipaa_text'].unique()
test_corpus = {i: text for i, text in enumerate(test_hipaa_texts)}
test_corpus_list = list(test_corpus.values())
corpus_embeddings = best_model.encode(test_corpus_list, convert_to_tensor=True)

recall_1 = recall_3 = recall_5 = mrr = 0
total_queries = 0

for _, row in test.iterrows():
    query_text = row['nist_text']
    true_text = row['hipaa_text']

    # Find corpus id of the true HIPAA text
    true_id = None
    for idx, ctext in test_corpus.items():
        if ctext == true_text:
            true_id = idx
            break
    if true_id is None:
        continue

    query_embed = best_model.encode(query_text, convert_to_tensor=True)
    cos_scores = torch.nn.functional.cosine_similarity(query_embed, corpus_embeddings)
    sorted_indices = torch.argsort(cos_scores, descending=True)
    rank = (sorted_indices == true_id).nonzero(as_tuple=True)[0].item() + 1

    if rank <= 1: recall_1 += 1
    if rank <= 3: recall_3 += 1
    if rank <= 5: recall_5 += 1
    mrr += 1 / rank
    total_queries += 1

recall_1 /= total_queries
recall_3 /= total_queries
recall_5 /= total_queries
mrr /= total_queries

print(f"Test queries evaluated: {total_queries}")
print(f"Recall@1: {recall_1:.4f}")
print(f"Recall@3: {recall_3:.4f}")
print(f"Recall@5: {recall_5:.4f}")
print(f"MRR:      {mrr:.4f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Test queries evaluated: 6
Recall@1: 0.1667
Recall@3: 0.5000
Recall@5: 1.0000
MRR:      0.3944


In [11]:
results = {
    'model': 'all-MiniLM-L6-v2 + MultipleNegativesRankingLoss',
    'positive_pairs': len(crosswalk),
    'recall@1': recall_1,
    'recall@3': recall_3,
    'recall@5': recall_5,
    'mrr': mrr,
    'date': str(datetime.date.today())
}
with open(MODEL_DIR / 'evaluation_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("✅ Evaluation results saved to", MODEL_DIR / 'evaluation_results.json')

✅ Evaluation results saved to ..\models\regmap-embedder\evaluation_results.json
